In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as transforms
from torchvision.utils import make_grid, save_image
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
import os
import cv2
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
 
# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
 
print("✓ All imports successful")
print(f"✓ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  - Device {i}: {torch.cuda.get_device_name(i)}")
    print(f"✓ Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

✓ All imports successful
✓ CUDA Available: True
✓ GPU Count: 2
  - Device 0: Tesla T4
  - Device 1: Tesla T4
✓ Memory Available: 15.64 GB


# Hyper-params initialization

In [2]:
class Config:
    """Configuration class for all hyperparameters"""
    
    # Device Configuration
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    MULTI_GPU = torch.cuda.device_count() > 1
    
    # Image Configuration
    IMG_HEIGHT = 256
    IMG_WIDTH = 256
    IMG_CHANNELS = 3  # RGB
    
    # Training Configuration
    BATCH_SIZE = 16
    EPOCHS = 100
    LEARNING_RATE = 0.0002
    BETA1 = 0.5
    BETA2 = 0.999
    LAMBDA_L1 = 100  # L1 loss weight
    
    # Mixed Precision
    USE_MIXED_PRECISION = True
    
    # Checkpoint Configuration
    CHECKPOINT_INTERVAL = 5
    SAVE_INTERVAL = 10
    
    # Paths
    BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle') else './results'
    CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints')
    RESULTS_DIR = os.path.join(BASE_DIR, 'results')
    LOGS_DIR = os.path.join(BASE_DIR, 'logs')
    
    # Create directories
    for directory in [CHECKPOINT_DIR, RESULTS_DIR, LOGS_DIR]:
        os.makedirs(directory, exist_ok=True)
    
    # Dataset Configuration
    TRAIN_SPLIT = 0.8
    VAL_SPLIT = 0.1
    TEST_SPLIT = 0.1
    
CONFIG = Config()
print(" Configuration loaded")
print(f"  Base Dir: {CONFIG.BASE_DIR}")
print(f"  Device: {CONFIG.DEVICE}")
print(f"  Multi-GPU: {CONFIG.MULTI_GPU}")

 Configuration loaded
  Base Dir: /kaggle/working
  Device: cuda
  Multi-GPU: True


# Dataset

In [3]:
class PairedImageDataset(Dataset):
    """Dataset for paired input-output images"""
    
    def __init__(self, input_paths, target_paths, img_height=256, img_width=256, use_augmentation=False):
        self.input_paths = input_paths
        self.target_paths = target_paths
        self.img_height = img_height
        self.img_width = img_width
        self.use_augmentation = use_augmentation
        
        assert len(input_paths) == len(target_paths), \
            f"Number of input ({len(input_paths)}) and target ({len(target_paths)}) images must match"
    
    def __len__(self):
        return len(self.input_paths)
    
    def __getitem__(self, idx):
        try:
            # Load images
            input_img = Image.open(self.input_paths[idx]).convert('RGB')
            target_img = Image.open(self.target_paths[idx]).convert('RGB')
            
            # Resize
            input_img = input_img.resize((self.img_width, self.img_height), Image.LANCZOS)
            target_img = target_img.resize((self.img_width, self.img_height), Image.LANCZOS)
            
            # Data augmentation (random flip for training)
            if self.use_augmentation and np.random.rand() > 0.5:
                input_img = input_img.transpose(Image.FLIP_LEFT_RIGHT)
                target_img = target_img.transpose(Image.FLIP_LEFT_RIGHT)
            
            # Convert to numpy
            input_array = np.array(input_img, dtype=np.float32)
            target_array = np.array(target_img, dtype=np.float32)
            
            # Normalize to [-1, 1]
            input_array = (input_array / 127.5) - 1.0
            target_array = (target_array / 127.5) - 1.0
            
            # Convert to tensor
            input_tensor = torch.from_numpy(input_array).permute(2, 0, 1)
            target_tensor = torch.from_numpy(target_array).permute(2, 0, 1)
            
            return {
                'input': input_tensor,
                'target': target_tensor,
                'input_path': self.input_paths[idx],
                'target_path': self.target_paths[idx]
            }
        
        except Exception as e:
            print(f"Error loading images at index {idx}: {e}")
            dummy = torch.zeros(3, self.img_height, self.img_width)
            return {'input': dummy, 'target': dummy, 'input_path': '', 'target_path': ''}
 
print("PairedImageDataset class defined")

PairedImageDataset class defined


# Generator

In [4]:
class ConvBlock(nn.Module):
    """Convolutional block with optional batch normalization and activation"""
    
    def __init__(self, in_channels, out_channels, kernel_size=4, stride=2, 
                 padding=1, activation='relu', use_bn=True):
        super(ConvBlock, self).__init__()
        
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, 
                     stride=stride, padding=padding, bias=False if use_bn else True)
        ]
        
        if use_bn:
            layers.append(nn.BatchNorm2d(out_channels))
        
        if activation == 'relu':
            layers.append(nn.ReLU(inplace=True))
        elif activation == 'leaky_relu':
            layers.append(nn.LeakyReLU(0.2, inplace=True))
        
        self.block = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.block(x)

class DeconvBlock(nn.Module):
    """Deconvolutional block with batch normalization"""
    
    def __init__(self, in_channels, out_channels, kernel_size=4, stride=2, 
                 padding=1, activation='relu', use_dropout=False):
        super(DeconvBlock, self).__init__()
        
        layers = [
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=kernel_size,
                             stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels)
        ]
        
        if use_dropout:
            layers.append(nn.Dropout(0.5))
        
        if activation == 'relu':
            layers.append(nn.ReLU(inplace=True))
        
        self.block = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.block(x)

print(" ConvBlock and DeconvBlock defined")

 ConvBlock and DeconvBlock defined


# generator

In [6]:
class Generator(nn.Module):
    """U-Net Generator with skip connections (FIXED)"""

    def __init__(self, in_channels=3, out_channels=3, base_filters=64):
        super(Generator, self).__init__()
        self.base_filters = base_filters

        # ENCODER (6 levels, not 7) - 256→128→64→32→16→8→4
        self.enc1 = ConvBlock(in_channels, base_filters, activation='leaky_relu', use_bn=False)
        self.enc2 = ConvBlock(base_filters, base_filters*2)
        self.enc3 = ConvBlock(base_filters*2, base_filters*4)
        self.enc4 = ConvBlock(base_filters*4, base_filters*8)
        self.enc5 = ConvBlock(base_filters*8, base_filters*8)
        self.enc6 = ConvBlock(base_filters*8, base_filters*8)

        # BOTTLENECK (stride=1, NO downsampling) - stays at 4×4
        self.bottleneck = nn.Sequential(
            nn.Conv2d(base_filters*8, base_filters*8, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True)
        )

        # DECODER (6 levels with correct concatenation channels)
        self.dec6 = DeconvBlock(base_filters*8, base_filters*8, use_dropout=True)
        self.dec5 = DeconvBlock(base_filters*16, base_filters*8, use_dropout=True)
        self.dec4 = DeconvBlock(base_filters*16, base_filters*8, use_dropout=True)
        self.dec3 = DeconvBlock(base_filters*16, base_filters*4)
        self.dec2 = DeconvBlock(base_filters*8, base_filters*2)
        self.dec1 = DeconvBlock(base_filters*4, base_filters)

        # Final output layer
        self.final = nn.Sequential(
            nn.Conv2d(base_filters*2, out_channels, kernel_size=1, stride=1, padding=0),
            nn.Tanh()
        )

    def forward(self, x):
        # ENCODER
        enc1 = self.enc1(x)           # [B, 64, 128, 128]
        enc2 = self.enc2(enc1)        # [B, 128, 64, 64]
        enc3 = self.enc3(enc2)        # [B, 256, 32, 32]
        enc4 = self.enc4(enc3)        # [B, 512, 16, 16]
        enc5 = self.enc5(enc4)        # [B, 512, 8, 8]
        enc6 = self.enc6(enc5)        # [B, 512, 4, 4]

        # BOTTLENECK
        bottleneck = self.bottleneck(enc6)  # [B, 512, 4, 4]

        # DECODER with skip connections (all dimensions match!)
        dec6 = self.dec6(bottleneck)        # [B, 512, 8, 8]
        dec6 = torch.cat([dec6, enc5], dim=1)  # [B, 1024, 8, 8] ✓

        dec5 = self.dec5(dec6)              # [B, 512, 16, 16]
        dec5 = torch.cat([dec5, enc4], dim=1)  # [B, 1024, 16, 16] ✓

        dec4 = self.dec4(dec5)              # [B, 512, 32, 32]
        dec4 = torch.cat([dec4, enc3], dim=1)  # [B, 768, 32, 32] ✓

        dec3 = self.dec3(dec4)              # [B, 256, 64, 64]
        dec3 = torch.cat([dec3, enc2], dim=1)  # [B, 384, 64, 64] ✓

        dec2 = self.dec2(dec3)              # [B, 128, 128, 128]
        dec2 = torch.cat([dec2, enc1], dim=1)  # [B, 192, 128, 128] ✓

        dec1 = self.dec1(dec2)              # [B, 64, 256, 256]
        dec1 = torch.cat([dec1, x], dim=1)  # [B, 67, 256, 256] ✓

        output = self.final(dec1)           # [B, 3, 256, 256]
        return output

# Test Generator
try:
    gen = Generator()
    test_input = torch.randn(4, 3, 256, 256)
    test_output = gen(test_input)
    print(f" GENERATOR TEST PASSED")
    print(f"  Input:  {test_input.shape}")
    print(f"  Output: {test_output.shape}")
except Exception as e:
    print(f" Generator error: {e}")
    import traceback
    traceback.print_exc()

 Generator error: Sizes of tensors must match except in dimension 1. Expected size 6 but got size 8 for tensor number 1 in the list.


Traceback (most recent call last):
  File "/tmp/ipykernel_55/1363791620.py", line 74, in <cell line: 0>
    test_output = gen(test_input)
                  ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_55/1363791620.py", line 50, in forward
    dec6 = torch.cat([dec6, enc5], dim=1)  # [B, 1024, 8, 8] ✓
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 6 but got size 8 for tensor number 1 in the list.


# Discriminator (Patch GAN)

In [5]:
class Discriminator(nn.Module):
    """PatchGAN Discriminator - Classifies 16×16 patches as real or fake"""
    
    def __init__(self, in_channels=6, base_filters=64):
        super(Discriminator, self).__init__()
        
        self.model = nn.Sequential(
            # 256 -> 128
            nn.Conv2d(in_channels, base_filters, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 128 -> 64
            nn.Conv2d(base_filters, base_filters*2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_filters*2),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 64 -> 32
            nn.Conv2d(base_filters*2, base_filters*4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_filters*4),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 32 -> 16
            nn.Conv2d(base_filters*4, base_filters*8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_filters*8),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 16 -> 16 (final patch classification)
            nn.Conv2d(base_filters*8, 1, kernel_size=4, stride=1, padding=1)
        )
    
    def forward(self, x, y):
        """Concatenate input and target, return patch probabilities"""
        combined = torch.cat([x, y], dim=1)
        return self.model(combined)

# Test Discriminator
try:
    disc = Discriminator()
    test_input = torch.randn(4, 3, 256, 256)
    test_target = torch.randn(4, 3, 256, 256)
    test_output = disc(test_input, test_target)
    print(f" DISCRIMINATOR TEST PASSED")
    print(f"  Input:  {test_input.shape}")
    print(f"  Target: {test_target.shape}")
    print(f"  Output: {test_output.shape} (patch classification matrix)")
except Exception as e:
    print(f" Discriminator error: {e}")

Discriminator network defined
Discriminator test passed. Output shape: torch.Size([1, 1, 15, 15])


# Loss Function

In [ ]:
# CELL 8: LOSS FUNCTIONS
class Pix2PixLoss(nn.Module):
    """Combined loss function: Adversarial + L1 Reconstruction"""
    
    def __init__(self, lambda_l1=100, device='cpu'):
        super(Pix2PixLoss, self).__init__()
        self.lambda_l1 = lambda_l1
        self.device = device
        self.bce_loss = nn.BCEWithLogitsLoss()
        self.l1_loss = nn.L1Loss()
    
    def discriminator_loss(self, disc_real_output, disc_fake_output):
        """D loss: classify real pairs as 1, fake pairs as 0"""
        real_target = torch.ones_like(disc_real_output)
        real_loss = self.bce_loss(disc_real_output, real_target)
        
        fake_target = torch.zeros_like(disc_fake_output)
        fake_loss = self.bce_loss(disc_fake_output, fake_target)
        
        return real_loss + fake_loss
    
    def generator_loss(self, disc_fake_output, generated_output, target_output):
        """G loss: Adversarial + λ * L1"""
        fake_target = torch.ones_like(disc_fake_output)
        adversarial_loss = self.bce_loss(disc_fake_output, fake_target)
        
        l1_loss = self.l1_loss(generated_output, target_output)
        total_loss = adversarial_loss + self.lambda_l1 * l1_loss
        
        return total_loss, adversarial_loss, l1_loss

print("Loss functions defined")

# evaluation metrics

In [ ]:
class Metrics:
    """Image quality metrics"""
    
    @staticmethod
    def ssim(img1, img2):
        """Structural Similarity Index (SSIM)"""
        # Convert to numpy
        if isinstance(img1, torch.Tensor):
            img1 = img1.cpu().numpy()
        if isinstance(img2, torch.Tensor):
            img2 = img2.cpu().numpy()
        
        # Normalize to [0, 1]
        img1 = np.clip((img1 + 1) / 2, 0, 1)
        img2 = np.clip((img2 + 1) / 2, 0, 1)
        
        # Convert to uint8
        img1 = (img1 * 255).astype(np.uint8)
        img2 = (img2 * 255).astype(np.uint8)
        if len(img1.shape) == 3:
            img1 = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
        if len(img2.shape) == 3:
            img2 = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)
        
        c1, c2 = 0.01, 0.03
        mean1 = cv2.blur(img1, (11, 11)).astype(np.float32)
        mean2 = cv2.blur(img2, (11, 11)).astype(np.float32)
        mean1_sq = mean1 ** 2
        mean2_sq = mean2 ** 2
        
        sigma1_sq = cv2.blur(img1.astype(np.float32)**2, (11, 11)) - mean1_sq
        sigma2_sq = cv2.blur(img2.astype(np.float32)**2, (11, 11)) - mean2_sq
        sigma12 = cv2.blur((img1.astype(np.float32) * img2.astype(np.float32)), (11, 11)) - mean1*mean2
        
        ssim_map = ((2*mean1*mean2 + c1) * (2*sigma12 + c2)) / ((mean1_sq + mean2_sq + c1) * (sigma1_sq + sigma2_sq + c2))
        return np.mean(ssim_map)
    
    @staticmethod
    def psnr(img1, img2):
        """Peak Signal-to-Noise Ratio"""
        if isinstance(img1, torch.Tensor):
            img1 = img1.cpu().numpy()
        if isinstance(img2, torch.Tensor):
            img2 = img2.cpu().numpy()
        
        img1 = np.clip((img1 + 1) / 2, 0, 1)
        img2 = np.clip((img2 + 1) / 2, 0, 1)
        
        mse = np.mean((img1 - img2) ** 2)
        if mse == 0:
            return 100
        
        psnr = 20 * np.log10(1.0 / np.sqrt(mse))
        return psnr
    
    @staticmethod
    def batch_ssim(generated, target):
        """SSIM for batch"""
        values = []
        for i in range(min(generated.size(0), 5)):
            val = Metrics.ssim(generated[i], target[i])
            values.append(val)
        return np.mean(values) if values else 0
    
    @staticmethod
    def batch_psnr(generated, target):
        """PSNR for batch"""
        values = []
        for i in range(min(generated.size(0), 5)):
            val = Metrics.psnr(generated[i], target[i])
            values.append(val)
        return np.mean(values) if values else 0

print("✓ Metrics class defined")

# visualization

In [ ]:
# CELL 10: VISUALIZATION UTILITIES
class Visualizer:
    """Visualization utility"""
    
    def __init__(self, save_dir):
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)
    
    @staticmethod
    def denormalize(tensor):
        """Convert [-1, 1] to [0, 1]"""
        return (tensor + 1) / 2
    
    def plot_samples(self, inputs, generated, targets, num_samples=5, save_name='samples.png'):
        """Plot input, generated, and target side by side"""
        num_samples = min(num_samples, inputs.size(0))
        fig = plt.figure(figsize=(15, 3*num_samples))
        gs = GridSpec(num_samples, 3, figure=fig, hspace=0.3, wspace=0.2)
        
        inputs = self.denormalize(inputs).cpu()
        generated = self.denormalize(generated).cpu()
        targets = self.denormalize(targets).cpu()
        
        for i in range(num_samples):
            # Input
            ax = fig.add_subplot(gs[i, 0])
            img = inputs[i].permute(1, 2, 0).numpy()
            ax.imshow(np.clip(img, 0, 1))
            ax.set_title('Input')
            ax.axis('off')
            
            # Generated
            ax = fig.add_subplot(gs[i, 1])
            img = generated[i].permute(1, 2, 0).numpy()
            ax.imshow(np.clip(img, 0, 1))
            ax.set_title('Generated')
            ax.axis('off')
            
            # Target
            ax = fig.add_subplot(gs[i, 2])
            img = targets[i].permute(1, 2, 0).numpy()
            ax.imshow(np.clip(img, 0, 1))
            ax.set_title('Target')
            ax.axis('off')
        
        save_path = os.path.join(self.save_dir, save_name)
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
        plt.close()
        return save_path
    
    def plot_losses(self, losses, save_name='training_losses.png'):
        """Plot training losses"""
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        
        axes[0, 0].plot(losses['gen_total'], linewidth=2, color='blue')
        axes[0, 0].set_title('Generator Total Loss')
        axes[0, 0].grid(True, alpha=0.3)
        
        axes[0, 1].plot(losses['gen_adversarial'], linewidth=2, color='orange')
        axes[0, 1].set_title('Generator Adversarial Loss')
        axes[0, 1].grid(True, alpha=0.3)
        
        axes[1, 0].plot(losses['gen_l1'], linewidth=2, color='green')
        axes[1, 0].set_title('Generator L1 Loss')
        axes[1, 0].grid(True, alpha=0.3)
        
        axes[1, 1].plot(losses['disc'], linewidth=2, color='red')
        axes[1, 1].set_title('Discriminator Loss')
        axes[1, 1].grid(True, alpha=0.3)
        
        fig.suptitle('Training Losses', fontsize=16, fontweight='bold')
        plt.tight_layout()
        save_path = os.path.join(self.save_dir, save_name)
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
        plt.close()
        return save_path

print(" Visualizer class defined")

# pix2pix trainer

In [ ]:
class Pix2PixTrainer:
    """Complete trainer for Pix2Pix"""
    
    def __init__(self, config):
        self.config = config
        self.device = config.DEVICE
        
        # Initialize models
        self.generator = Generator(in_channels=3, out_channels=3)
        self.discriminator = Discriminator(in_channels=6)
        
        # Multi-GPU support
        if config.MULTI_GPU:
            self.generator = nn.DataParallel(self.generator)
            self.discriminator = nn.DataParallel(self.discriminator)
        
        # Move to device
        self.generator = self.generator.to(self.device)
        self.discriminator = self.discriminator.to(self.device)
        
        # Optimizers
        self.gen_optimizer = optim.Adam(
            self.generator.parameters(),
            lr=config.LEARNING_RATE,
            betas=(config.BETA1, config.BETA2)
        )
        
        self.disc_optimizer = optim.Adam(
            self.discriminator.parameters(),
            lr=config.LEARNING_RATE,
            betas=(config.BETA1, config.BETA2)
        )
        
        # Loss function
        self.loss_fn = Pix2PixLoss(lambda_l1=config.LAMBDA_L1, device=self.device)
        
        # Mixed precision
        self.use_amp = config.USE_MIXED_PRECISION
        self.scaler = GradScaler() if self.use_amp else None
        
        # Tracking
        self.losses = {
            'gen_total': [],
            'gen_adversarial': [],
            'gen_l1': [],
            'disc': []
        }
        
        self.best_metrics = {
            'ssim': 0,
            'psnr': 0,
            'epoch': -1
        }
    
    def train_step(self, batch):
        """Single training step"""
        input_imgs = batch['input'].to(self.device)
        target_imgs = batch['target'].to(self.device)
        
        # ===== DISCRIMINATOR TRAINING =====
        self.disc_optimizer.zero_grad()
        
        with autocast(enabled=self.use_amp):
            with torch.no_grad():
                generated_imgs = self.generator(input_imgs)
            
            disc_real_output = self.discriminator(input_imgs, target_imgs)
            disc_fake_output = self.discriminator(input_imgs, generated_imgs.detach())
            
            d_loss = self.loss_fn.discriminator_loss(disc_real_output, disc_fake_output)
        
        if self.use_amp:
            self.scaler.scale(d_loss).backward()
            self.scaler.step(self.disc_optimizer)
            self.scaler.update()
        else:
            d_loss.backward()
            self.disc_optimizer.step()
        
        # ===== GENERATOR TRAINING =====
        self.gen_optimizer.zero_grad()
        
        with autocast(enabled=self.use_amp):
            generated_imgs = self.generator(input_imgs)
            disc_fake_output = self.discriminator(input_imgs, generated_imgs)
            
            g_total_loss, g_adv_loss, g_l1_loss = self.loss_fn.generator_loss(
                disc_fake_output, generated_imgs, target_imgs
            )
        
        if self.use_amp:
            self.scaler.scale(g_total_loss).backward()
            self.scaler.step(self.gen_optimizer)
            self.scaler.update()
        else:
            g_total_loss.backward()
            self.gen_optimizer.step()
        
        return {
            'gen_total': g_total_loss.item(),
            'gen_adversarial': g_adv_loss.item(),
            'gen_l1': g_l1_loss.item(),
            'disc': d_loss.item()
        }
    
    def train_epoch(self, train_loader, epoch):
        """Train for one epoch"""
        self.generator.train()
        self.discriminator.train()
        
        epoch_losses = {k: 0.0 for k in self.losses.keys()}
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{self.config.EPOCHS}")
        
        for batch_idx, batch in enumerate(pbar):
            losses = self.train_step(batch)
            
            for key in epoch_losses.keys():
                epoch_losses[key] += losses[key]
            
            pbar.set_postfix({
                'G_Total': losses['gen_total']:.4f,
                'G_L1': losses['gen_l1']:.4f,
                'D': losses['disc']:.4f
            })
        
        # Average losses
        for key in epoch_losses.keys():
            epoch_losses[key] /= len(train_loader)
            self.losses[key].append(epoch_losses[key])
        
        return epoch_losses
    
    def save_checkpoint(self, epoch, is_best=False):
        """Save checkpoint"""
        checkpoint = {
            'epoch': epoch,
            'generator': self.generator.state_dict(),
            'discriminator': self.discriminator.state_dict(),
            'gen_optimizer': self.gen_optimizer.state_dict(),
            'disc_optimizer': self.disc_optimizer.state_dict(),
            'losses': self.losses,
            'best_metrics': self.best_metrics
        }
        
        # Save regular checkpoint
        path = os.path.join(
            self.config.CHECKPOINT_DIR,
            f'pix2pix_epoch_{epoch:03d}.pth'
        )
        torch.save(checkpoint, path)
        
        # Save best model
        if is_best:
            best_path = os.path.join(self.config.CHECKPOINT_DIR, 'best_model.pth')
            torch.save(checkpoint, best_path)
            print(f"  ★ Best model saved: {best_path}")
        
        return path
    
    def load_checkpoint(self, checkpoint_path):
        """Load checkpoint"""
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        
        self.generator.load_state_dict(checkpoint['generator'])
        self.discriminator.load_state_dict(checkpoint['discriminator'])
        self.gen_optimizer.load_state_dict(checkpoint['gen_optimizer'])
        self.disc_optimizer.load_state_dict(checkpoint['disc_optimizer'])
        self.losses = checkpoint['losses']
        
        return checkpoint['epoch']

print(" Pix2PixTrainer class defined")

# dataset

In [ ]:
# CELL 12: LOAD KAGGLE DATASETS
def load_kaggle_datasets(config):
    """Load paired datasets from Kaggle input directory"""
    
    # Try Face Sketch Dataset first
    face_sketch_path = '/kaggle/input/datasets/arbazkhan971/cuhk-face-sketch-database-cufs'
    anime_dataset_path = '/kaggle/input/datasets/ktaebum/anime-sketch-colorization-pair'
    
    input_paths, target_paths = [], []
    dataset_name = None
    
    # Check Face Sketch Dataset
    if os.path.exists(face_sketch_path):
        print(f"\n✓ Found Face Sketch Dataset at {face_sketch_path}")
        # Adapt to actual structure - typically sketch/photo pairs
        sketch_dir = os.path.join(face_sketch_path, 'sketch')  # Modify path as needed
        photo_dir = os.path.join(face_sketch_path, 'photo')    # Modify path as needed
        
        if os.path.exists(sketch_dir) and os.path.exists(photo_dir):
            sketch_files = sorted([f for f in os.listdir(sketch_dir) if f.endswith(('.jpg', '.png'))])
            photo_files = sorted([f for f in os.listdir(photo_dir) if f.endswith(('.jpg', '.png'))])
            
            for sketch_f, photo_f in zip(sketch_files, photo_files):
                input_paths.append(os.path.join(sketch_dir, sketch_f))
                target_paths.append(os.path.join(photo_dir, photo_f))
            
            dataset_name = 'Face Sketch'
    
    # Check Anime Dataset
    elif os.path.exists(anime_dataset_path):
        print(f"\n✓ Found Anime Dataset at {anime_dataset_path}")
        # Adapt to actual structure
        sketch_dir = os.path.join(anime_dataset_path, 'sketch')
        color_dir = os.path.join(anime_dataset_path, 'color')
        
        if os.path.exists(sketch_dir) and os.path.exists(color_dir):
            sketch_files = sorted([f for f in os.listdir(sketch_dir) if f.endswith(('.jpg', '.png'))])
            color_files = sorted([f for f in os.listdir(color_dir) if f.endswith(('.jpg', '.png'))])
            
            for sketch_f, color_f in zip(sketch_files, color_files):
                input_paths.append(os.path.join(sketch_dir, sketch_f))
                target_paths.append(os.path.join(color_dir, color_f))
            
            dataset_name = 'Anime Colorization'
    
    if not input_paths:
        print("\n⚠ No datasets found. Using dummy dataset for testing.")
        input_paths, target_paths = PairedImageDataset.get_dummy_dataset(size=100)
        dataset_name = 'Dummy (Testing)'
    
    print(f"\n✓ Dataset: {dataset_name}")
    print(f"✓ Total pairs: {len(input_paths)}")
    
    return input_paths, target_paths, dataset_name

# Add dummy dataset method
PairedImageDataset.get_dummy_dataset = staticmethod(lambda size=100: (
    [f'/tmp/dummy_{i}_input.jpg' for i in range(size)],
    [f'/tmp/dummy_{i}_target.jpg' for i in range(size)]
))

print("✓ Dataset loading functions defined")

# data loading

In [ ]:
# CELL 13: CREATE DATALOADERS
def create_dataloaders(input_paths, target_paths, config, val_split=0.2):
    """Create train and validation dataloaders"""
    
    # Filter valid paths
    valid_pairs = []
    for inp, tgt in zip(input_paths, target_paths):
        if os.path.exists(inp) and os.path.exists(tgt):
            valid_pairs.append((inp, tgt))
    
    if not valid_pairs:
        raise ValueError("No valid image pairs found!")
    
    input_paths_valid = [p[0] for p in valid_pairs]
    target_paths_valid = [p[1] for p in valid_pairs]
    
    # Split data
    total_samples = len(input_paths_valid)
    val_size = int(total_samples * val_split)
    train_size = total_samples - val_size
    
    train_input = input_paths_valid[:train_size]
    train_target = target_paths_valid[:train_size]
    val_input = input_paths_valid[train_size:]
    val_target = target_paths_valid[train_size:]
    
    # Create datasets
    train_dataset = PairedImageDataset(
        train_input, train_target,
        img_height=config.IMG_HEIGHT,
        img_width=config.IMG_WIDTH,
        use_augmentation=True
    )
    
    val_dataset = PairedImageDataset(
        val_input, val_target,
        img_height=config.IMG_HEIGHT,
        img_width=config.IMG_WIDTH,
        use_augmentation=False
    )
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )
    
    print(f"Train samples: {train_size}")
    print(f"Val samples: {val_size}")
    
    return train_loader, val_loader

print("DataLoader creation functions defined")

# training loop

In [ ]:
# CELL 14: MAIN TRAINING LOOP
def train_pix2pix(config, train_loader, val_loader):
    """Main training function"""
    
    trainer = Pix2PixTrainer(config)
    visualizer = Visualizer(config.RESULTS_DIR)
    metrics_history = {}
    
    print("\n" + "="*70)
    print("STARTING PIX2PIX TRAINING")
    print("="*70)
    print(f"Device: {config.DEVICE}")
    print(f"Epochs: {config.EPOCHS}")
    print(f"Batch Size: {config.BATCH_SIZE}")
    print(f"Train samples: {len(train_loader.dataset)}")
    print(f"Val samples: {len(val_loader.dataset)}")
    print("="*70 + "\n")
    
    try:
        for epoch in range(config.EPOCHS):
            # Training
            train_losses = trainer.train_epoch(train_loader, epoch)
            
            # Validation
            trainer.generator.eval()
            trainer.discriminator.eval()
            
            val_ssim_list, val_psnr_list = [], []
            
            with torch.no_grad():
                for batch in tqdm(val_loader, desc=f"Validating", leave=False):
                    input_imgs = batch['input'].to(config.DEVICE)
                    target_imgs = batch['target'].to(config.DEVICE)
                    
                    generated = trainer.generator(input_imgs)
                    
                    ssim = Metrics.batch_ssim(generated, target_imgs)
                    psnr = Metrics.batch_psnr(generated, target_imgs)
                    
                    val_ssim_list.append(ssim)
                    val_psnr_list.append(psnr)
            
            val_ssim = np.mean(val_ssim_list) if val_ssim_list else 0
            val_psnr = np.mean(val_psnr_list) if val_psnr_list else 0
            
            metrics_history[epoch] = {
                'train_loss': train_losses['gen_total'],
                'val_ssim': val_ssim,
                'val_psnr': val_psnr
            }
            
            # Update best metrics
            is_best = False
            if val_ssim > trainer.best_metrics['ssim']:
                trainer.best_metrics['ssim'] = val_ssim
                trainer.best_metrics['psnr'] = val_psnr
                trainer.best_metrics['epoch'] = epoch
                is_best = True
            
            # Print summary every 5 epochs
            if (epoch + 1) % 5 == 0:
                print(f"\nEpoch {epoch+1}/{config.EPOCHS}")
                print(f"  Train: G_Total={train_losses['gen_total']:.4f}, "
                      f"G_L1={train_losses['gen_l1']:.4f}, D={train_losses['disc']:.4f}")
                print(f"  Val:   SSIM={val_ssim:.4f}, PSNR={val_psnr:.4f}")
            
            # Save checkpoints
            if (epoch + 1) % config.CHECKPOINT_INTERVAL == 0:
                trainer.save_checkpoint(epoch, is_best=is_best)
            
            # Always save best model
            if is_best:
                trainer.save_checkpoint(epoch, is_best=True)
            
            # Visualize every 10 epochs
            if (epoch + 1) % config.SAVE_INTERVAL == 0:
                with torch.no_grad():
                    sample_batch = next(iter(val_loader))
                    sample_input = sample_batch['input'].to(config.DEVICE)
                    sample_target = sample_batch['target'].to(config.DEVICE)
                    sample_generated = trainer.generator(sample_input)
                    
                    visualizer.plot_samples(
                        sample_input, sample_generated, sample_target,
                        num_samples=5,
                        save_name=f'epoch_{epoch+1:03d}.png'
                    )
                    print(f"  ✓ Samples saved")
    
    except KeyboardInterrupt:
        print("\n Training interrupted")
    except Exception as e:
        print(f"\n✗ Training error: {e}")
        import traceback
        traceback.print_exc()
    
    # Final visualizations
    print("\n" + "="*70)
    print("GENERATING FINAL VISUALIZATIONS")
    print("="*70)
    
    visualizer.plot_losses(trainer.losses)
    print(" Loss plots saved")
    
    best_path = os.path.join(config.CHECKPOINT_DIR, 'best_model.pth')
    print(f"\n Best model: {best_path}")
    print(f"Best SSIM: {trainer.best_metrics['ssim']:.4f}")
    print(f"Best PSNR: {trainer.best_metrics['psnr']:.4f}")
    print("="*70 + "\n")
    
    return trainer, visualizer, metrics_history

print("Training function defined")

In [ ]:
print("\n" + "="*70)
print("LOADING DATASETS")
print("="*70)

input_paths, target_paths, dataset_name = load_kaggle_datasets(CONFIG)

# Create dataloaders
train_loader, val_loader = create_dataloaders(input_paths, target_paths, CONFIG)

# Train
trainer, visualizer, metrics_history = train_pix2pix(CONFIG, train_loader, val_loader)

# testing

In [ ]:
# CELL 16: INFERENCE CLASS
class Pix2PixInference:
    """Inference on single images"""
    
    def __init__(self, checkpoint_path, device='cuda'):
        self.device = torch.device(device)
        
        self.generator = Generator()
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        
        # Handle DataParallel
        state_dict = checkpoint['generator']
        if 'module.' in list(state_dict.keys())[0]:
            state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
        
        self.generator.load_state_dict(state_dict)
        self.generator = self.generator.to(self.device)
        self.generator.eval()
        print(f"✓ Model loaded from {checkpoint_path}")
    
    def predict(self, input_image_path):
        """Generate output from input"""
        img = Image.open(input_image_path).convert('RGB')
        img = img.resize((256, 256), Image.LANCZOS)
        img_array = np.array(img, dtype=np.float32)
        
        img_array = (img_array / 127.5) - 1.0
        img_tensor = torch.from_numpy(img_array).permute(2, 0, 1).unsqueeze(0)
        img_tensor = img_tensor.to(self.device)
        
        with torch.no_grad():
            output = self.generator(img_tensor)
        
        output = (output + 1) / 2
        output = output.squeeze(0).cpu().permute(1, 2, 0).numpy()
        output = np.clip(output, 0, 1)
        
        return output

print("Inference class defined")